# 00 · Project Overview

**Competition:** [RSNA STR Pulmonary Embolism Detection](https://www.kaggle.com/competitions/rsna-str-pulmonary-embolism-detection/overview) (Kaggle, 2020)

This project rebuilds a solution to the challenge from scratch and scores it with late submissions. The notebooks are meant to be read in order.

## The competition

- **Host:** RSNA with the Society of Thoracic Radiology (STR).
- **Dates:** September 10 – October 26, 2020. A code competition: predictions are made in a Kaggle notebook, offline, within 9 GPU hours.
- **Scale:** 784 teams; $30,000 in prizes.

A pulmonary embolism (PE) is a clot blocking an artery in the lungs. It is diagnosed on CT pulmonary angiography (CTPA), where the clot shows as a dark gap inside a contrast-filled artery. The challenge asked for a model that finds PE and describes it.

## The task

**Input: one exam = one 3D CT scan of the chest.** Each of its 512 × 512 grayscale images is a cross-section of the body, called a **slice**; stacked in order they form a 3D volume. Pixel values are Hounsfield units (HU), a fixed density scale (air −1000, water 0, bone +1000).

**Output: probabilities of yes/no labels, at two levels.**
- **Slice level (2D):** is a clot visible on this slice? One label per slice.
- **Exam level (3D):** nine labels about the whole scan.

So the task is **binary classification with several labels, on 2D slices and on 3D exams**.

| Label | Level | Meaning |
|---|---|---|
| `pe_present_on_image` | slice | a clot is visible on this slice |
| `negative_exam_for_pe` | exam | no clot anywhere in the scan |
| `indeterminate` | exam | the scan is too poor to decide (patient moved, contrast too weak) |
| `rightsided_pe`, `leftsided_pe` | exam | a clot in the arteries of the right / left lung |
| `central_pe` | exam | a clot in the main pulmonary arteries, next to the heart |
| `rv_lv_ratio_gte_1` | exam | the right heart chamber is at least as wide as the left one: the clot is straining the heart |
| `rv_lv_ratio_lt_1` | exam | the right chamber is narrower than the left one |
| `chronic_pe`, `acute_and_chronic_pe` | exam | the clot is old, or old and new together; neither means new (acute) |

**The exam labels follow fixed rules.** Every exam is exactly one of negative, indeterminate or positive. A positive exam has at least one location label, exactly one of the two RV/LV labels and at most one type label; negative and indeterminate exams have none of these.

**Why the slice level is not enough.** A slice label can be decided from that slice and its neighbours. The exam labels cannot: RV/LV is measured on the heart, usually on slices without any clot, and the location labels combine clots seen across the whole scan. Every top solution therefore had two stages: a 2D CNN on each slice (with the neighbouring slices as extra channels), then a model that reads the CNN outputs of all slices in order and predicts the exam labels.

## Evaluation metric

A **weighted log loss over exams and slices**. Exam $i$ has $n_i$ slices, $m_i$ of them positive, $q_i = m_i / n_i$:

$$\text{score} = \frac{\sum_i \Big[ \sum_{j=1}^{9} w_j \, \ell(y_{ij}, p_{ij}) + \sum_{k=1}^{n_i} w \, q_i \, \ell(y_{ik}, p_{ik}) \Big]}{\sum_i \Big[ \sum_{j=1}^{9} w_j + \sum_{k=1}^{n_i} w \, q_i \Big]}, \qquad \ell(y, p) = -\big[ y \log p + (1 - y) \log (1 - p) \big]$$

| Label | Weight |
|---|---:|
| `rv_lv_ratio_gte_1` | 0.2347 |
| `central_pe` | 0.1877 |
| `chronic_pe`, `acute_and_chronic_pe` | 0.1043 each |
| `indeterminate` | 0.0920 |
| `rv_lv_ratio_lt_1` | 0.0782 |
| `negative_exam_for_pe` | 0.0736 |
| `rightsided_pe`, `leftsided_pe` | 0.0626 each |
| `pe_present_on_image` | $w = 0.0736 \times q_i$ per slice |

- **Slices count only in positive exams** ($q_i = 0$ otherwise), and they are **49% of the total weight** in train.
- `rv_lv_ratio_gte_1` and `central_pe` are 42% of the exam weight.
- Predictions must obey the label hierarchy at a 0.5 threshold; we enforce it in every submission (06).

## Data

| Split | Exams | Slices |
|-------|------:|-------:|
| Train | 7,279 | 1,790,594 |
| Test, visible | 650 | 146,853 |
| Test, full (Kaggle only) | 2,167 | 532,091 |

- RSPECT dataset ([Colak et al. 2021](https://pubs.rsna.org/doi/full/10.1148/ryai.2021200254)), five institutions in five countries. One exam per patient. 980 GB.
- **PE is common per exam, rare per slice:** 30.4% of training exams, 5.4% of slices.
- **Train exams had one reader, test exams three** (with adjudication). Validation scores are therefore much worse than the leaderboard (1st place: about 0.18 locally, 0.140 private); they rank models, and the real level comes from late submissions.